# Prophet Validation — LabOps Agent

Validates Prophet forecast accuracy on synthetic TSH demand data calibrated with real Argentine lab patterns.

**Output:** `prophet_metrics.json`

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics

# Add backend to path
sys.path.insert(0, os.path.join('..', 'backend'))
import prediction

## 1. Load synthetic demand history (TSH)

Uses the same `_build_synthetic_history` function as the production backend.

In [ ]:
REAGENT = 'TSH'
HISTORY_DAYS = 365

df = prediction._build_synthetic_history(REAGENT, days=HISTORY_DAYS)
print(f'Shape: {df.shape}')
print(df.head())
print(f'\nDate range: {df.ds.min()} to {df.ds.max()}')

## 2. Train Prophet model

In [ ]:
m = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    interval_width=0.8,
)
m.fit(df)
print('Model trained.')

## 3. Cross-validation

In [ ]:
df_cv = cross_validation(
    m,
    initial='180 days',
    period='30 days',
    horizon='14 days',
    parallel='processes'
)
print(f'CV rows: {len(df_cv)}')
print(df_cv.head())

## 4. Performance metrics

In [ ]:
df_p = performance_metrics(df_cv)
print(df_p[['horizon', 'mae', 'rmse', 'mape']].head())

## 5. Aggregate metrics

In [ ]:
metrics = {
    'reagent': REAGENT,
    'history_days': HISTORY_DAYS,
    'cv_initial': '180 days',
    'cv_period': '30 days',
    'cv_horizon': '14 days',
    'mae': round(float(df_p['mae'].mean()), 2),
    'rmse': round(float(df_p['rmse'].mean()), 2),
    'mape': round(float(df_p['mape'].mean()), 4),
    'mape_percent': round(float(df_p['mape'].mean()) * 100, 2),
    'coverage': round(float(df_p['coverage'].mean()), 4) if 'coverage' in df_p.columns else None,
    'calibrated': True,
    'note': 'DEMO — synthetic data calibrated with real demand patterns',
}

print('=== Prophet Performance Metrics ===')
print(f'MAE  : {metrics["mae"]} units')
print(f'RMSE : {metrics["rmse"]} units')
print(f'MAPE : {metrics["mape_percent"]}%')
print(f'Coverage : {metrics["coverage"]}')

# Save to JSON
out_path = 'prophet_metrics.json'
with open(out_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'\nSaved to {out_path}')